# Transfection results

Reads `analysis/` written by `analyze.ipynb` and writes user-facing `results/<sample>/` packs. Tables (`*.xlsx`) and Plots (`*.png`) are separate cells so you can regenerate figures without rewriting spreadsheets.

This notebook owns sample names and merges them into `workspace/assay.json` as `samples[]` only. It does not write `interval`, `analysis.channels`, or `maxOnsetMinutes` (those come from `analyze.ipynb`). No Config signal channel. Run `analyze.ipynb` first.


## Config

In [ ]:
from pathlib import Path
import json

# Folder with analysis/ (from analyze.ipynb); results/<sample>/ is written here. Not the ND2/CZI file.
WORKSPACE = Path(r"Z:\\projects\LNPbinder\Experiments\20260731\Auswertung")

# Minutes between acquired frames (must match analyze.ipynb). 10.0 → t = 0, 10, 20, …
# Used for plot time axes. Interval in assay.json is written by analyze.ipynb, not here.
INTERVAL_MINUTES = 10.0

# One entry per sample. name is the folder under results/ (use filesystem-safe names).
# positions: zero-based field indices, same as roi/Pos{n} and analyze.
#   list(range(0, 40)) is positions 0 through 39 (Python half-open range), not 0..40 inclusive.
SAMPLES = [
    {"name": "A431_aiLNP_incubated", "positions": list(range(0, 40))},
    {"name": "A549_aiLNP_incubated", "positions": list(range(40, 80))},
    {"name": "A549_aiLNP", "positions": list(range(80, 121))},
    {"name": "A431_aiLNP", "positions": list(range(121, 159))},
]


def _inclusive_position_spec(positions):
    ordered = sorted({int(position) for position in positions})
    if not ordered:
        raise ValueError("No positions to serialize")
    parts = []
    start = prev = ordered[0]
    for value in ordered[1:]:
        if value == prev + 1:
            prev = value
            continue
        parts.append(f"{start}:{prev}" if start != prev else str(start))
        start = prev = value
    parts.append(f"{start}:{prev}" if start != prev else str(start))
    return ",".join(parts)


workspace = WORKSPACE.expanduser().resolve()
if not workspace.is_dir():
    raise FileNotFoundError(f"Workspace not found: {workspace}")
assay_path = workspace / "assay.json"
payload = {}
if assay_path.is_file():
    payload = json.loads(assay_path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError(f"{assay_path} must contain a JSON object")
samples_payload = []
for slide_channel, row in enumerate(SAMPLES):
    name = str(row.get("name", "")).strip()
    positions = row.get("positions")
    samples_payload.append(
        {
            "slideChannel": slide_channel,
            "name": name,
            "positions": _inclusive_position_spec(positions),
        }
    )
payload["samples"] = samples_payload
assay_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
print(f"Workspace: {workspace}")
print(f"Merged assay.json samples: {assay_path}")


## Tables

In [ ]:
from transfection.core import (
    load_assay_for_workspace,
    publish_sample_tables_xlsx,
    publish_sample_traces_xlsx,
    require_named_samples,
)

mapping = require_named_samples(load_assay_for_workspace(workspace))
for path in publish_sample_traces_xlsx(workspace, mapping):
    print(f"Wrote table: {path}")
for path in publish_sample_tables_xlsx(workspace, mapping, "auc"):
    print(f"Wrote table: {path}")
for path in publish_sample_tables_xlsx(workspace, mapping, "fit"):
    print(f"Wrote table: {path}")


## Plots

Re-run Plots without re-running Tables to regenerate figures.


In [ ]:
from transfection.services import plot_auc, plot_fit, plot_timeseries

for path in plot_timeseries.run_plot_timeseries(
    metrics_dir=workspace,
    interval=INTERVAL_MINUTES,
):
    print(plot_timeseries.format_written_timeseries_plot_message(path))

for message in plot_auc.format_written_auc_plot_messages(
    list(plot_auc.run_plot_auc(auc_csv=workspace))
):
    print(message)

for message in plot_fit.format_written_fit_plot_messages(
    plot_fit.run_plot_fit(
        workspace,
        output=None,
        interval=INTERVAL_MINUTES,
        columns=None,
    )
):
    print(message)

print("Results plots finished.")
